# Mutual information in the doorway grid

These experiments call the authors' actual occupancy and mutual-information code.
The source checkout is expected at `~/projects/geometry-of-empowerment`.
Run with `uv run --with jupyterlab jupyter lab examples/mutual_information_lab.ipynb`
from that checkout. All logarithms below are natural; units are **nats**.

Change the starting cell, discount, and search budget to investigate the geometry.
The random vertex search gives a lower bound on potential empowerment, not an exact global capacity certificate.


In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.home() / "projects" / "geometry-of-empowerment"
sys.path.insert(0, str(REPO))
from environments import GridEnv, two_room_walls
from utils.blahut_arimoto import build_skill_data
from utils.information import mutual_information, kl_divergence

print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## Control versus randomness

Both channels below have the same 50/50 marginal outcome distribution.
In the first channel the selected skill determines the outcome. In the second,
the outcome is independent of the skill. Only the first carries information.


In [ ]:
prior = np.array([0.5, 0.5])
channels = {
    "Skill determines left/right": np.eye(2),
    "Both skills produce the same random outcome": np.full((2, 2), 0.5),
}
for name, channel in channels.items():
    mi = mutual_information(channel, prior)
    print(f"{name}: {mi:.6f} nats = {mi / np.log(2):.3f} bits; marginal = {prior @ channel}")


## Choose a starting state and time horizon

The central doorway is between columns 3 and 4 at row 4. Coordinates are zero-based.
`gamma=0.5` emphasizes the immediate future; `gamma=0.95` looks further ahead.
Start with the doorway cell `(4, 3)`, then compare an interior cell such as `(4, 1)`.


In [ ]:
GAMMA = 0.95
START = (4, 3)
DIRECTIONS = 500
SEED = 0

env = GridEnv(rows=8, cols=8, walls=two_room_walls(rows=8, wall_col=3, hole_row=4))
s0 = env.to_s(*START)
data = build_skill_data(env, GAMMA, start_states=[s0], n_directions=DIRECTIONS, seed=SEED)
channel = data.pz_s0[s0]
prior = data.p_prior[s0]
mixture = prior @ channel
print("Discovered candidate skills:", len(prior))
print("Effective MI / capacity lower bound:", mutual_information(channel, prior), "nats")


## Calculate the same MI in two ways

Knowing which skill ran reduces uncertainty about the sampled future state.
Equivalently, MI averages each skill's KL divergence from the shared mixture.
The sampled state comes from the discounted occupancy, including the initial state.


In [ ]:
def entropy(p):
    p = np.asarray(p)
    return -(p * np.log(np.maximum(p, 1e-300))).sum(axis=-1)

h_future = entropy(mixture)
h_given_skill = prior @ entropy(channel)
mi_from_entropy = h_future - h_given_skill
mi_from_kl = prior @ kl_divergence(channel, mixture)
np.testing.assert_allclose(mi_from_entropy, mi_from_kl, atol=1e-10)
print(f"H(S+) = {h_future:.6f}")
print(f"H(S+ | Z) = {h_given_skill:.6f}")
print(f"I(Z; S+) = {mi_from_entropy:.6f} nats = {mi_from_entropy / np.log(2):.6f} bits")


## Inspect one skill and the shared mixture

Change `TARGET` to choose a skill that puts high occupancy on another grid cell.
The two plots share one probability color scale. A point in occupancy geometry
represents an entire heatmap like one of these.


In [ ]:
TARGET = (1, 6)
active = np.flatnonzero(prior > 1e-6)
z = active[np.argmax(channel[active, env.to_s(*TARGET)])]
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for ax, p, label in zip(axes, [mixture, channel[z]], ["Mixture q(s)", f"Skill {z}: p(s | z)"]):
    image = ax.imshow(p.reshape(8, 8), vmin=0, vmax=channel[z].max(), cmap="viridis")
    for (row, col), _ in env.walls:
        ax.plot([col+0.5, col+0.5], [row-0.5, row+0.5], color="black", linewidth=2)
    ax.scatter([START[1]], [START[0]], marker="*", s=120, color="white", edgecolors="black")
    ax.set_title(label)
fig.colorbar(image, ax=axes, label="Discounted occupancy probability")
plt.show()


## Keep the same future diversity, remove skill control

Blend each conditional distribution with the shared mixture:

`new_channel = alpha * channel + (1 - alpha) * mixture`

At `alpha=1`, use the original skills. At `alpha=0`, every skill produces exactly
the same mixture. The marginal distribution remains unchanged for every alpha,
while MI drops to zero. The original optimized skill prior stays fixed here;
we are measuring effective MI, not re-optimizing capacity at every alpha.


In [ ]:
alphas = np.linspace(0, 1, 51)
values = []
for alpha in alphas:
    blended = alpha * channel + (1-alpha) * mixture[None, :]
    np.testing.assert_allclose(prior @ blended, mixture, atol=1e-12)
    values.append(mutual_information(blended, prior))
assert abs(values[0]) < 1e-10
plt.figure(figsize=(7, 4))
plt.plot(alphas, values, linewidth=2)
plt.xlabel("Skill-specific contribution, alpha")
plt.ylabel("Mutual information (nats)")
plt.title("Same marginal future, different controllability")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Connection to Craftax and a learned world model

The grid code has a known transition model. A learned world model could supply
predicted future distributions for candidate skills, but its prediction errors
could also exaggerate distinguishability. A useful later experiment is to compare
MI estimated from real simulator rollouts with MI estimated from model rollouts,
using the same initial states, skills, representation, and observation-time rule.
This notebook does not train a Craftax policy or claim to reproduce the companion scaling results.
